# MVSR – Vorbereitung Einheit 6
## Machine Learning: Regression, Clustering und PCA

**Bearbeitungszeit:** ca. 45–60 Minuten

Dieses Notebook bereitet die Inhalte der sechsten MVSR-Einheit praktisch vor. In den bisherigen Einheiten wurden Merkmale, Korrespondenzen und geometrische Verfahren weitgehend **manuell modelliert**. Jetzt betrachten wir die zentrale Idee des Machine Learnings:

> **Abhängigkeiten und Repräsentationen werden aus Daten gelernt.**

Der praktische Schwerpunkt liegt auf drei Themen:

- **Supervised Learning:** lineare Regression, Loss und Training,
- **Unsupervised Learning:** k-Means Clustering,
- **Statistische Merkmale:** Principal Component Analysis (PCA).

Für dieses Binder-Notebook wird **C++17 mit OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- Supervised, Unsupervised und Reinforcement Learning grundsätzlich unterscheiden,
- Klassifikation und Regression auseinanderhalten,
- ein einfaches lineares Regressionsmodell formulieren,
- Residuen und Mean Squared Error berechnen,
- ein Modell mit Gradient Descent trainieren,
- den Unterschied zwischen Training und Anwendung erklären,
- beschreiben, was mit **Generalisierung** gemeint ist,
- k-Means Clustering durchführen,
- PCA als lineare Projektion in einen niedrigdimensionalen Raum erklären,
- Kovarianzmatrix, Eigenwerte und Eigenvektoren mit PCA in Verbindung bringen,
- hochdimensionale Bilddaten mit PCA reduzieren und visualisieren.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook wird über **MyBinder** direkt im Browser ausgeführt. Der vorhandene C++-Code kann unmittelbar in den Notebook-Zellen bearbeitet und ausgeführt werden; eine zusätzliche lokale Installation ist dafür nicht erforderlich.

Arbeiten Sie das Notebook möglichst **von oben nach unten** durch. Spätere Zellen verwenden Variablen und Ergebnisse aus früheren Abschnitten.

Bei Aufgaben mit `TODO` sollen Sie Parameter selbstständig verändern und deren Einfluss auf das Ergebnis beobachten.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Dadurch können bereits deklarierte globale Variablen bei erneutem Ausführen einer Zelle zu einer Fehlermeldung führen. Die `TODO`-Zellen sind deshalb wieder teilweise in lokale Lambda-Funktionen gekapselt.

> **Wichtig:** Änderungen innerhalb einer Binder-Sitzung werden nicht dauerhaft im GitHub-Repository gespeichert.

## 0. Setup

Für diese C++-Version verwenden wir bewusst nur **OpenCV 4.6** und die C++-Standardbibliothek.

Damit benötigen wir in MyBinder keine zusätzlichen Machine-Learning-Bibliotheken. Die Python-Version verwendet teilweise `scikit-learn`; die entsprechenden Schritte werden hier mit OpenCV bzw. direkt in C++ umgesetzt.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind spezifisch für den interaktiven C++-Kernel. In einem normalen lokalen C++-Projekt wird OpenCV über das Build-System, z.B. mit CMake, eingebunden und gelinkt.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <algorithm>
#include <numeric>
#include <random>
#include <limits>
#include <cstdlib>
#include <sstream>

#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Bilddarstellung

Die Python-Version verwendet Matplotlib für Diagramme. Im C++-Binder-Notebook erzeugen wir die Visualisierungen direkt als `cv::Mat` und zeigen sie im Browser an.

In einem normalen lokalen C++-Programm würde man für Bilder typischerweise `cv::imshow()` und `cv::waitKey()` verwenden.

> **Binder-/Notebook-Hinweis:** `show_image(...)` muss als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

### Allgemeine Plot-Hilfsfunktion

Damit die eigentlichen Machine-Learning-Beispiele übersichtlich bleiben, übernimmt die folgende Hilfsfunktion die Darstellung einfacher 2D-Daten.

> **Binder-/Notebook-Hinweis:** Die Plot-Funktion ersetzt hier lediglich Matplotlib. Sie ist nicht Teil des Machine-Learning-Inhalts.

In [ ]:
cv::Mat render_xy_plot(
    const std::vector<cv::Point2d>& points,
    const std::vector<cv::Point2d>& line_points,
    const std::vector<std::pair<cv::Point2d, cv::Point2d>>& segments,
    const std::string& title,
    const std::string& x_label,
    const std::string& y_label
)
{
    const int width = 760;
    const int height = 520;
    const int left = 80;
    const int right = 30;
    const int top = 45;
    const int bottom = 70;

    std::vector<cv::Point2d> all_points = points;

    all_points.insert(
        all_points.end(),
        line_points.begin(),
        line_points.end()
    );

    for (const auto& segment : segments)
    {
        all_points.push_back(segment.first);
        all_points.push_back(segment.second);
    }

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    if (all_points.empty())
    {
        return canvas;
    }

    double min_x = all_points.front().x;
    double max_x = all_points.front().x;
    double min_y = all_points.front().y;
    double max_y = all_points.front().y;

    for (const auto& p : all_points)
    {
        min_x = std::min(min_x, p.x);
        max_x = std::max(max_x, p.x);
        min_y = std::min(min_y, p.y);
        max_y = std::max(max_y, p.y);
    }

    double range_x = std::max(max_x - min_x, 1e-9);
    double range_y = std::max(max_y - min_y, 1e-9);

    min_x -= 0.08 * range_x;
    max_x += 0.08 * range_x;
    min_y -= 0.10 * range_y;
    max_y += 0.10 * range_y;

    range_x = max_x - min_x;
    range_y = max_y - min_y;

    auto to_pixel =
        [&](const cv::Point2d& p)
        {
            int x = cvRound(
                left
                + (p.x - min_x)
                / range_x
                * (width - left - right)
            );

            int y = cvRound(
                height - bottom
                - (p.y - min_y)
                / range_y
                * (height - top - bottom)
            );

            return cv::Point(x, y);
        };

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    for (const auto& segment : segments)
    {
        cv::line(
            canvas,
            to_pixel(segment.first),
            to_pixel(segment.second),
            cv::Scalar(150, 150, 150),
            1,
            cv::LINE_AA
        );
    }

    if (line_points.size() >= 2)
    {
        for (std::size_t i = 1; i < line_points.size(); ++i)
        {
            cv::line(
                canvas,
                to_pixel(line_points[i - 1]),
                to_pixel(line_points[i]),
                cv::Scalar(200, 90, 20),
                2,
                cv::LINE_AA
            );
        }
    }

    for (const auto& p : points)
    {
        cv::circle(
            canvas,
            to_pixel(p),
            5,
            cv::Scalar(40, 40, 40),
            -1,
            cv::LINE_AA
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(20, 28),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(0, 0, 0),
        2
    );

    cv::putText(
        canvas,
        x_label,
        cv::Point(width / 2 - 100, height - 20),
        cv::FONT_HERSHEY_SIMPLEX,
        0.55,
        cv::Scalar(0, 0, 0),
        1
    );

    cv::putText(
        canvas,
        y_label,
        cv::Point(8, 52),
        cv::FONT_HERSHEY_SIMPLEX,
        0.5,
        cv::Scalar(0, 0, 0),
        1
    );

    return canvas;
}

### Hilfsfunktion für Downloads

Für den PCA-Abschnitt mit echten Bilddaten verwenden wir später das offizielle OpenCV-Testbild `digits.png`.

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \"" + url + "\" -O \"" + output_file + "\"";

    int result = std::system(command.c_str());

    if (result != 0)
    {
        std::cerr << "Download fehlgeschlagen: "
                  << url << std::endl;

        return false;
    }

    return true;
}

## 1. Machine Learning in a Nutshell

Bisher wurden viele Funktionen explizit vorgegeben:

- Kantenfilter,
- Harris Corner Detector,
- SIFT,
- geometrische Projektion,
- PnP.

Beim Machine Learning wird stattdessen versucht, eine Abbildung aus **Daten** zu lernen.

**Daten → Modell → gelernte Abbildung**

| Lernart | Information beim Lernen | Typische Aufgabe |
|---|---|---|
| **Supervised Learning** | Eingabe $X$ + Label $y$ | Klassifikation, Regression |
| **Unsupervised Learning** | nur Eingabe $X$ | Clustering, Dimensionsreduktion |
| **Reinforcement Learning** | Interaktion + Reward | Steuerung, Navigation, Manipulation |

In diesem Notebook liegt der Schwerpunkt auf **Supervised** und **Unsupervised Learning**.

### Klassifikation vs. Regression

Beim Supervised Learning wird eine Abbildung $f: X \mapsto y$ aus gelabelten Trainingsdaten gelernt.

- **Klassifikation:** $y$ ist diskret, z.B. `Hund`, `Katze`, `Auto`.
- **Regression:** $y$ ist kontinuierlich, z.B. Preis, Temperatur oder Abstand.

Wir starten mit Regression, weil sich daran Modell, Loss und Training besonders anschaulich erklären lassen.

## 2. Supervised Learning – Regression

Wir verwenden das Beispiel aus der Python-Version:

**Wohnfläche → Verkaufspreis**

In [ ]:
std::vector<double> area = {
    45, 55, 60, 70, 90,
    100, 110, 115, 135, 145, 155
};

std::vector<double> price = {
    200, 165, 210, 280, 350,
    300, 380, 470, 520, 580, 610
};

std::vector<cv::Point2d> house_points;

for (std::size_t i = 0; i < area.size(); ++i)
{
    house_points.emplace_back(
        area[i],
        price[i]
    );
}

cv::Mat regression_data_plot =
    render_xy_plot(
        house_points,
        {},
        {},
        "Beobachtete Verkaeufe",
        "Wohnflaeche [m^2]",
        "Preis [Tsd. Euro]"
    );

In [ ]:
show_image(regression_data_plot)

### Lineares Modell

Ein einfaches lineares Regressionsmodell lautet:

$\hat{y} = wx + b$

mit

- $x$: Eingabe,
- $\hat{y}$: Vorhersage,
- $w$: Steigung,
- $b$: Achsenabschnitt.

In [ ]:
double w = 2.0;
double b = 50.0;

std::vector<double> prediction(
    area.size()
);

std::vector<cv::Point2d>
    prediction_line;

for (std::size_t i = 0; i < area.size(); ++i)
{
    prediction[i] =
        w * area[i] + b;

    prediction_line.emplace_back(
        area[i],
        prediction[i]
    );
}

cv::Mat initial_model_plot =
    render_xy_plot(
        house_points,
        prediction_line,
        {},
        "Aktuelles lineares Modell",
        "Wohnflaeche [m^2]",
        "Preis [Tsd. Euro]"
    );

In [ ]:
show_image(initial_model_plot)

## 3. Residuum und Loss

Für einen Datenpunkt ist das **Residuum**

$e_i = y_i - \hat{y}_i$.

Eine häufig verwendete Loss-Funktion für Regression ist der **Mean Squared Error**:

$L = \frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2$

Durch das Quadrieren

- heben sich positive und negative Fehler nicht gegenseitig auf,
- große Fehler werden stärker gewichtet.

In [ ]:
std::vector<double> residuals(
    area.size()
);

double mse = 0.0;

for (std::size_t i = 0; i < area.size(); ++i)
{
    residuals[i] =
        price[i] - prediction[i];

    mse +=
        residuals[i] * residuals[i];
}

mse /= area.size();

std::cout << "Residuen:" << std::endl;

for (double residual : residuals)
{
    std::cout << std::fixed
              << std::setprecision(2)
              << residual
              << " ";
}

std::cout << "\n\nMean Squared Error: "
          << mse
          << std::endl;

In [ ]:
std::vector<
    std::pair<cv::Point2d, cv::Point2d>
> residual_segments;

for (std::size_t i = 0; i < area.size(); ++i)
{
    residual_segments.push_back({
        cv::Point2d(
            area[i],
            price[i]
        ),
        cv::Point2d(
            area[i],
            prediction[i]
        )
    });
}

cv::Mat residual_plot =
    render_xy_plot(
        house_points,
        prediction_line,
        residual_segments,
        "Residuen des aktuellen Modells",
        "Wohnflaeche [m^2]",
        "Preis [Tsd. Euro]"
    );

In [ ]:
show_image(residual_plot)

## 4. Wie lernt das Modell?

Gradient Descent aktualisiert die Parameter in Richtung eines kleineren Loss:

$w \leftarrow w - \eta \frac{\partial L}{\partial w}$

$b \leftarrow b - \eta \frac{\partial L}{\partial b}$

$\eta$ ist die **Learning Rate**.

Damit die numerischen Werte beim Training gut skaliert sind, normalisieren wir die Wohnfläche.

In [ ]:
double area_mean =
    std::accumulate(
        area.begin(),
        area.end(),
        0.0
    ) / area.size();

double area_variance = 0.0;

for (double value : area)
{
    double d =
        value - area_mean;

    area_variance += d * d;
}

area_variance /= area.size();

double area_std =
    std::sqrt(area_variance);

std::vector<double> x(
    area.size()
);

std::vector<double> y =
    price;

for (std::size_t i = 0; i < area.size(); ++i)
{
    x[i] =
        (area[i] - area_mean)
        / area_std;
}

std::cout << "Mittelwert: "
          << area_mean
          << std::endl;

std::cout << "Standardabweichung: "
          << area_std
          << std::endl;

### Trainingsfunktion

> **xeus-cling-Hinweis:** Die Trainingsfunktion steht bewusst alleine in einer Codezelle. Das ist für den inkrementellen C++-Interpreter robuster als mehrere freie Funktionsdefinitionen in derselben Zelle.

In [ ]:
void train_linear_regression(
    const std::vector<double>& x,
    const std::vector<double>& y,
    double learning_rate,
    int epochs,
    double& w,
    double& b,
    std::vector<double>& loss_history
)
{
    w = 0.0;

    b =
        std::accumulate(
            y.begin(),
            y.end(),
            0.0
        ) / y.size();

    loss_history.clear();

    for (int epoch = 0;
         epoch < epochs;
         ++epoch)
    {
        double loss = 0.0;
        double dL_dw = 0.0;
        double dL_db = 0.0;

        for (std::size_t i = 0;
             i < x.size();
             ++i)
        {
            double y_hat =
                w * x[i] + b;

            double error =
                y_hat - y[i];

            loss +=
                error * error;

            dL_dw +=
                error * x[i];

            dL_db +=
                error;
        }

        loss /=
            x.size();

        dL_dw *=
            2.0 / x.size();

        dL_db *=
            2.0 / x.size();

        loss_history.push_back(
            loss
        );

        w -=
            learning_rate * dL_dw;

        b -=
            learning_rate * dL_db;
    }
}

In [ ]:
double w_trained = 0.0;
double b_trained = 0.0;

std::vector<double>
    loss_history;

train_linear_regression(
    x,
    y,
    0.05,
    300,
    w_trained,
    b_trained,
    loss_history
);

std::cout
    << "Gelernter Parameter w: "
    << w_trained
    << std::endl;

std::cout
    << "Gelernter Parameter b: "
    << b_trained
    << std::endl;

std::cout
    << "Finaler Loss: "
    << loss_history.back()
    << std::endl;

In [ ]:
std::vector<cv::Point2d>
    loss_points;

for (std::size_t i = 0;
     i < loss_history.size();
     ++i)
{
    loss_points.emplace_back(
        static_cast<double>(i),
        loss_history[i]
    );
}

cv::Mat loss_plot =
    render_xy_plot(
        {},
        loss_points,
        {},
        "Training: Entwicklung des Loss",
        "Epoche",
        "MSE"
    );

In [ ]:
show_image(loss_plot)

In [ ]:
[]()
{
    std::vector<cv::Point2d> trained_line_local;

    trained_line_local.reserve(
        area.size()
    );

    for (std::size_t i = 0;
         i < area.size();
         ++i)
    {
        double y_hat =
            w_trained * x[i]
            + b_trained;

        trained_line_local.emplace_back(
            area[i],
            y_hat
        );
    }

    std::vector<
        std::pair<cv::Point2d, cv::Point2d>
    > no_segments;

    cv::Mat result =
        render_xy_plot(
            house_points,
            trained_line_local,
            no_segments,
            "Lineare Regression nach dem Training",
            "Wohnflaeche [m^2]",
            "Preis [Tsd. Euro]"
        );

    return show_image(result);
}()

### Mini-Aufgabe: Learning Rate

Verändern Sie die Learning Rate und beobachten Sie die Loss-Kurve.

Probieren Sie beispielsweise Werte zwischen `0.001` und `0.5`.

In [ ]:
[]()
{
    // TODO: Learning Rate verändern

    double learning_rate_test = 0.05;
    int epochs_test = 150;

    double w_test = 0.0;
    double b_test = 0.0;

    std::vector<double>
        loss_test;

    train_linear_regression(
        x,
        y,
        learning_rate_test,
        epochs_test,
        w_test,
        b_test,
        loss_test
    );

    std::cout
        << "Finaler Loss: "
        << loss_test.back()
        << std::endl;

    std::vector<cv::Point2d>
        plot_points;

    for (std::size_t i = 0;
         i < loss_test.size();
         ++i)
    {
        plot_points.emplace_back(
            static_cast<double>(i),
            loss_test[i]
        );
    }

    cv::Mat result =
        render_xy_plot(
            {},
            plot_points,
            {},
            "Learning Rate = "
                + std::to_string(
                    learning_rate_test
                ),
            "Epoche",
            "MSE"
        );

    return show_image(result);
}()

## 5. Training, Anwendung und Generalisierung

Während des **Trainings** sind Eingabe $x$ und Ground Truth $y$ bekannt:

**$x + y$ → Modelltraining**

Bei der späteren **Anwendung** ist nur $x$ bekannt:

**$x$ → trainiertes Modell → $\hat{y}$**

Ein Modell soll nicht nur die Trainingsdaten auswendig lernen, sondern auch auf **neue, bisher ungesehene Daten** sinnvoll reagieren. Diese Eigenschaft wird als **Generalisierung** bezeichnet.

Für den folgenden kleinen Test teilen wir die Daten reproduzierbar in Trainings- und Testdaten auf.

In [ ]:
std::vector<int> indices(
    area.size()
);

std::iota(
    indices.begin(),
    indices.end(),
    0
);

std::mt19937 split_rng(4);

std::shuffle(
    indices.begin(),
    indices.end(),
    split_rng
);

int test_count = 3;

std::vector<double>
    area_train;

std::vector<double>
    price_train;

std::vector<double>
    area_test;

std::vector<double>
    price_test;

for (std::size_t i = 0;
     i < indices.size();
     ++i)
{
    int index =
        indices[i];

    if (
        static_cast<int>(i)
        < test_count
    )
    {
        area_test.push_back(
            area[index]
        );

        price_test.push_back(
            price[index]
        );
    }
    else
    {
        area_train.push_back(
            area[index]
        );

        price_train.push_back(
            price[index]
        );
    }
}

double train_mean =
    std::accumulate(
        area_train.begin(),
        area_train.end(),
        0.0
    ) / area_train.size();

double train_variance = 0.0;

for (double value : area_train)
{
    double d =
        value - train_mean;

    train_variance +=
        d * d;
}

train_variance /=
    area_train.size();

double train_std =
    std::sqrt(
        train_variance
    );

std::vector<double>
    x_train(
        area_train.size()
    );

std::vector<double>
    x_test(
        area_test.size()
    );

for (std::size_t i = 0;
     i < area_train.size();
     ++i)
{
    x_train[i] =
        (area_train[i] - train_mean)
        / train_std;
}

for (std::size_t i = 0;
     i < area_test.size();
     ++i)
{
    x_test[i] =
        (area_test[i] - train_mean)
        / train_std;
}

In [ ]:
double w_split = 0.0;
double b_split = 0.0;

std::vector<double>
    split_loss;

train_linear_regression(
    x_train,
    price_train,
    0.05,
    300,
    w_split,
    b_split,
    split_loss
);

double train_mse = 0.0;
double test_mse = 0.0;

for (std::size_t i = 0;
     i < x_train.size();
     ++i)
{
    double error =
        price_train[i]
        - (
            w_split * x_train[i]
            + b_split
        );

    train_mse +=
        error * error;
}

train_mse /=
    x_train.size();

for (std::size_t i = 0;
     i < x_test.size();
     ++i)
{
    double error =
        price_test[i]
        - (
            w_split * x_test[i]
            + b_split
        );

    test_mse +=
        error * error;
}

test_mse /=
    x_test.size();

std::cout << "Train-MSE: "
          << train_mse
          << std::endl;

std::cout << "Test-MSE:  "
          << test_mse
          << std::endl;

## 6. Unsupervised Learning

Beim Supervised Learning besitzen wir Trainingsbeispiele der Form $(\mathbf{x},y)$.

Beim Unsupervised Learning besitzen wir dagegen nur $\mathbf{x}$.

Typische Ziele sind:

- **Clustering:** ähnliche Daten zu Gruppen zusammenfassen,
- **Dimensionsreduktion:** hochdimensionale Daten kompakter darstellen.

## 7. k-Means Clustering

Beim k-Means Clustering wird die Anzahl der Cluster $k$ vorgegeben.

Der Ablauf ist:

1. $k$ Clusterzentren initialisieren,
2. jeden Datenpunkt dem **nächstgelegenen Zentrum** zuordnen,
3. jedes Zentrum als Mittelwert seiner zugeordneten Punkte neu berechnen,
4. wiederholen.

> **Wichtig:** Bei **k-Means** steht $k$ für die **Anzahl der Cluster**. Das ist etwas anderes als **k-Nearest Neighbours (kNN)**, bei dem $k$ die Anzahl betrachteter Nachbarn bezeichnet.

In [ ]:
cv::Mat houses =
    (cv::Mat_<float>(12, 2)
        << 40, 150,
           45, 170,
           50, 180,
           55, 210,
           85, 320,
           95, 360,
           105, 390,
           110, 420,
           135, 540,
           145, 580,
           155, 640,
           165, 690);

std::vector<cv::Point2d>
    house_cluster_points;

for (int i = 0;
     i < houses.rows;
     ++i)
{
    house_cluster_points.emplace_back(
        houses.at<float>(i, 0),
        houses.at<float>(i, 1)
    );
}

cv::Mat cluster_input_plot =
    render_xy_plot(
        house_cluster_points,
        {},
        {},
        "Daten ohne bekannte Klassen",
        "Wohnflaeche [m^2]",
        "Preis [Tsd. Euro]"
    );

In [ ]:
show_image(cluster_input_plot)

### Feature-Skalierung

Die beiden Features besitzen unterschiedliche Zahlenbereiche.

Ohne Skalierung würde der Verkaufspreis aufgrund seiner größeren numerischen Werte die euklidische Distanz stärker beeinflussen.

In [ ]:
cv::Mat houses_scaled =
    houses.clone();

cv::Mat houses_mean =
    cv::Mat::zeros(
        1,
        2,
        CV_32F
    );

cv::Mat houses_std =
    cv::Mat::zeros(
        1,
        2,
        CV_32F
    );

for (int column = 0;
     column < 2;
     ++column)
{
    double mean_value = 0.0;

    for (int row = 0;
         row < houses.rows;
         ++row)
    {
        mean_value +=
            houses.at<float>(
                row,
                column
            );
    }

    mean_value /=
        houses.rows;

    double variance = 0.0;

    for (int row = 0;
         row < houses.rows;
         ++row)
    {
        double d =
            houses.at<float>(
                row,
                column
            )
            - mean_value;

        variance +=
            d * d;
    }

    variance /=
        houses.rows;

    double std_value =
        std::sqrt(
            variance
        );

    houses_mean.at<float>(
        0,
        column
    ) =
        static_cast<float>(
            mean_value
        );

    houses_std.at<float>(
        0,
        column
    ) =
        static_cast<float>(
            std_value
        );

    for (int row = 0;
         row < houses.rows;
         ++row)
    {
        houses_scaled.at<float>(
            row,
            column
        ) =
            (
                houses.at<float>(
                    row,
                    column
                )
                - mean_value
            )
            / std_value;
    }
}

std::cout << "Mittelwerte: "
          << houses_mean
          << std::endl;

std::cout << "Standardabweichungen: "
          << houses_std
          << std::endl;

### Hilfsfunktion für die Cluster-Darstellung

In [ ]:
cv::Mat render_cluster_plot(
    const cv::Mat& points,
    const cv::Mat& labels,
    const cv::Mat& centers,
    const std::string& title
)
{
    const int width = 760;
    const int height = 520;
    const int left = 80;
    const int right = 30;
    const int top = 45;
    const int bottom = 70;

    double min_x = points.at<float>(0, 0);
    double max_x = min_x;
    double min_y = points.at<float>(0, 1);
    double max_y = min_y;

    for (int i = 0; i < points.rows; ++i)
    {
        min_x = std::min(
            min_x,
            static_cast<double>(
                points.at<float>(i, 0)
            )
        );

        max_x = std::max(
            max_x,
            static_cast<double>(
                points.at<float>(i, 0)
            )
        );

        min_y = std::min(
            min_y,
            static_cast<double>(
                points.at<float>(i, 1)
            )
        );

        max_y = std::max(
            max_y,
            static_cast<double>(
                points.at<float>(i, 1)
            )
        );
    }

    double range_x =
        std::max(
            max_x - min_x,
            1.0
        );

    double range_y =
        std::max(
            max_y - min_y,
            1.0
        );

    min_x -=
        0.08 * range_x;

    max_x +=
        0.08 * range_x;

    min_y -=
        0.10 * range_y;

    max_y +=
        0.10 * range_y;

    range_x =
        max_x - min_x;

    range_y =
        max_y - min_y;

    auto to_pixel =
        [&](double x, double y)
        {
            int px = cvRound(
                left
                + (x - min_x)
                / range_x
                * (width - left - right)
            );

            int py = cvRound(
                height - bottom
                - (y - min_y)
                / range_y
                * (height - top - bottom)
            );

            return cv::Point(px, py);
        };

    std::vector<cv::Scalar> colors = {
        cv::Scalar(220, 70, 70),
        cv::Scalar(70, 150, 40),
        cv::Scalar(60, 90, 220),
        cv::Scalar(180, 80, 180),
        cv::Scalar(40, 170, 190),
        cv::Scalar(120, 120, 120)
    };

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    for (int i = 0;
         i < points.rows;
         ++i)
    {
        int label =
            labels.at<int>(i, 0);

        cv::circle(
            canvas,
            to_pixel(
                points.at<float>(i, 0),
                points.at<float>(i, 1)
            ),
            6,
            colors[
                label % colors.size()
            ],
            -1,
            cv::LINE_AA
        );
    }

    for (int i = 0;
         i < centers.rows;
         ++i)
    {
        cv::Point p =
            to_pixel(
                centers.at<float>(i, 0),
                centers.at<float>(i, 1)
            );

        cv::line(
            canvas,
            p + cv::Point(-8, -8),
            p + cv::Point(8, 8),
            cv::Scalar(0, 0, 0),
            3,
            cv::LINE_AA
        );

        cv::line(
            canvas,
            p + cv::Point(-8, 8),
            p + cv::Point(8, -8),
            cv::Scalar(0, 0, 0),
            3,
            cv::LINE_AA
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(20, 28),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(0, 0, 0),
        2
    );

    return canvas;
}

In [ ]:
int k = 3;

cv::Mat labels;
cv::Mat centers_scaled;

cv::TermCriteria kmeans_criteria(
    cv::TermCriteria::EPS
        + cv::TermCriteria::MAX_ITER,
    100,
    1e-4
);

cv::kmeans(
    houses_scaled,
    k,
    labels,
    kmeans_criteria,
    10,
    cv::KMEANS_PP_CENTERS,
    centers_scaled
);

cv::Mat centers =
    centers_scaled.clone();

for (int row = 0;
     row < centers.rows;
     ++row)
{
    for (int column = 0;
         column < 2;
         ++column)
    {
        centers.at<float>(
            row,
            column
        ) =
            centers_scaled.at<float>(
                row,
                column
            )
            * houses_std.at<float>(
                0,
                column
            )
            + houses_mean.at<float>(
                0,
                column
            );
    }
}

std::cout << "Clusterzuordnung:"
          << std::endl
          << labels.t()
          << std::endl;

std::cout << "\nClusterzentren:"
          << std::endl
          << centers
          << std::endl;

In [ ]:
cv::Mat cluster_plot =
    render_cluster_plot(
        houses,
        labels,
        centers,
        "k-Means Clustering"
    );

In [ ]:
show_image(cluster_plot)

### Mini-Aufgabe: Anzahl der Cluster

Verändern Sie `k_test`.

Probieren Sie beispielsweise `2`, `3`, `4` und `5`.

In [ ]:
[]()
{
    // TODO: Anzahl der Cluster verändern

    int k_test = 4;

    cv::Mat labels_test;
    cv::Mat centers_scaled_test;

    cv::kmeans(
        houses_scaled,
        k_test,
        labels_test,
        kmeans_criteria,
        10,
        cv::KMEANS_PP_CENTERS,
        centers_scaled_test
    );

    cv::Mat centers_test =
        centers_scaled_test.clone();

    for (int row = 0;
         row < centers_test.rows;
         ++row)
    {
        for (int column = 0;
             column < 2;
             ++column)
        {
            centers_test.at<float>(
                row,
                column
            ) =
                centers_scaled_test.at<float>(
                    row,
                    column
                )
                * houses_std.at<float>(
                    0,
                    column
                )
                + houses_mean.at<float>(
                    0,
                    column
                );
        }
    }

    cv::Mat result =
        render_cluster_plot(
            houses,
            labels_test,
            centers_test,
            "k-Means mit k = "
                + std::to_string(
                    k_test
                )
        );

    return show_image(result);
}()

### Eine k-Means-Iteration nachvollziehen

Um den Ablauf sichtbar zu machen, führen wir einen einzelnen Zuordnungs- und Update-Schritt manuell aus.

In [ ]:
cv::Mat centers_manual(
    3,
    2,
    CV_32F
);

houses_scaled.row(0).copyTo(
    centers_manual.row(0)
);

houses_scaled.row(5).copyTo(
    centers_manual.row(1)
);

houses_scaled.row(11).copyTo(
    centers_manual.row(2)
);

cv::Mat labels_manual(
    houses_scaled.rows,
    1,
    CV_32S
);

for (int i = 0;
     i < houses_scaled.rows;
     ++i)
{
    double best_distance =
        std::numeric_limits<double>::max();

    int best_cluster = -1;

    for (int cluster = 0;
         cluster < 3;
         ++cluster)
    {
        double dx =
            houses_scaled.at<float>(i, 0)
            - centers_manual.at<float>(
                cluster,
                0
            );

        double dy =
            houses_scaled.at<float>(i, 1)
            - centers_manual.at<float>(
                cluster,
                1
            );

        double distance =
            std::sqrt(
                dx * dx + dy * dy
            );

        if (distance < best_distance)
        {
            best_distance =
                distance;

            best_cluster =
                cluster;
        }
    }

    labels_manual.at<int>(
        i,
        0
    ) =
        best_cluster;
}

cv::Mat new_centers_manual =
    cv::Mat::zeros(
        3,
        2,
        CV_32F
    );

std::vector<int>
    cluster_counts(
        3,
        0
    );

for (int i = 0;
     i < houses_scaled.rows;
     ++i)
{
    int cluster =
        labels_manual.at<int>(
            i,
            0
        );

    new_centers_manual.at<float>(
        cluster,
        0
    ) +=
        houses_scaled.at<float>(
            i,
            0
        );

    new_centers_manual.at<float>(
        cluster,
        1
    ) +=
        houses_scaled.at<float>(
            i,
            1
        );

    cluster_counts[cluster]++;
}

for (int cluster = 0;
     cluster < 3;
     ++cluster)
{
    new_centers_manual.row(
        cluster
    ) /=
        static_cast<float>(
            cluster_counts[cluster]
        );
}

std::cout << "Alte Zentren:"
          << std::endl
          << centers_manual
          << std::endl;

std::cout << "\nNeue Zentren nach einer Iteration:"
          << std::endl
          << new_centers_manual
          << std::endl;

## 8. Principal Component Analysis (PCA)

PCA ist eine lineare Methode zur **Dimensionsreduktion**.

Ausgangslage:

$\mathbf{X}\in\mathbb{R}^{n\times D}$

Gesucht ist eine reduzierte Darstellung

$\mathbf{X}^{*}\in\mathbb{R}^{n\times d}$

Die Hauptkomponenten entsprechen den Eigenvektoren der Kovarianzmatrix.

Für die erste Hauptkomponente gilt:

$\boldsymbol{\Sigma}\mathbf{u}_1=\lambda_1\mathbf{u}_1$

Der größte Eigenwert $\lambda_1$ gehört zur Richtung mit der größten Varianz.

### PCA zunächst in 2D

In [ ]:
std::mt19937 pca_rng(4);

std::normal_distribution<double>
    pca_noise(
        0.0,
        0.55
    );

const int pca_samples = 80;

cv::Mat X(
    pca_samples,
    2,
    CV_64F
);

for (int i = 0;
     i < pca_samples;
     ++i)
{
    double x1 =
        -3.0
        + 6.0
        * i
        / (pca_samples - 1.0);

    double x2 =
        0.8 * x1
        + pca_noise(
            pca_rng
        );

    X.at<double>(i, 0) =
        x1;

    X.at<double>(i, 1) =
        x2;
}

std::vector<cv::Point2d>
    pca_points;

for (int i = 0;
     i < X.rows;
     ++i)
{
    pca_points.emplace_back(
        X.at<double>(i, 0),
        X.at<double>(i, 1)
    );
}

cv::Mat pca_input_plot =
    render_xy_plot(
        pca_points,
        {},
        {},
        "Korrelierte 2D-Daten",
        "Feature 1",
        "Feature 2"
    );

In [ ]:
show_image(pca_input_plot)

In [ ]:
cv::Mat mu;

cv::reduce(
    X,
    mu,
    0,
    cv::REDUCE_AVG,
    CV_64F
);

cv::Mat X_centered =
    X
    - cv::repeat(
        mu,
        X.rows,
        1
    );

cv::Mat covariance =
    (
        X_centered.t()
        * X_centered
    )
    / (X.rows - 1.0);

cv::Mat eigenvalues;
cv::Mat eigenvectors;

cv::eigen(
    covariance,
    eigenvalues,
    eigenvectors
);

std::cout << "Kovarianzmatrix:"
          << std::endl
          << covariance
          << std::endl;

std::cout << "\nEigenwerte:"
          << std::endl
          << eigenvalues
          << std::endl;

std::cout << "\nEigenvektoren:"
          << std::endl
          << eigenvectors
          << std::endl;

std::cout
    << "\nHinweis: cv::eigen speichert die Eigenvektoren zeilenweise."
    << std::endl;

In [ ]:
std::vector<
    std::pair<cv::Point2d, cv::Point2d>
> eigenvector_segments;

double pca_scale = 2.5;

for (int i = 0;
     i < 2;
     ++i)
{
    double scale =
        std::sqrt(
            eigenvalues.at<double>(
                i,
                0
            )
        )
        * pca_scale;

    cv::Point2d start(
        mu.at<double>(0, 0),
        mu.at<double>(0, 1)
    );

    cv::Point2d end(
        start.x
            + eigenvectors.at<double>(
                i,
                0
            ) * scale,
        start.y
            + eigenvectors.at<double>(
                i,
                1
            ) * scale
    );

    eigenvector_segments.push_back({
        start,
        end
    });
}

cv::Mat pca_axes_plot =
    render_xy_plot(
        pca_points,
        {},
        eigenvector_segments,
        "Hauptkomponenten",
        "Feature 1",
        "Feature 2"
    );

In [ ]:
show_image(pca_axes_plot)

### Reduktion von 2D auf 1D

Wir behalten nur die erste Hauptkomponente $\mathbf{u}_1$:

$\mathbf{X}^{*} = \mathbf{X}_{zentriert}\mathbf{u}_1$

In [ ]:
cv::Mat u1 =
    eigenvectors.row(
        0
    ).t();

cv::Mat X_1d =
    X_centered * u1;

std::cout
    << "Urspruengliche Form: "
    << X.rows
    << " x "
    << X.cols
    << std::endl;

std::cout
    << "Reduzierte Form:    "
    << X_1d.rows
    << " x "
    << X_1d.cols
    << std::endl;

In [ ]:
cv::Mat X_reconstructed =
    X_1d * u1.t()
    + cv::repeat(
        mu,
        X.rows,
        1
    );

std::vector<cv::Point2d>
    reconstructed_points;

for (int i = 0;
     i < X_reconstructed.rows;
     ++i)
{
    reconstructed_points.emplace_back(
        X_reconstructed.at<double>(
            i,
            0
        ),
        X_reconstructed.at<double>(
            i,
            1
        )
    );
}

std::vector<
    std::pair<cv::Point2d, cv::Point2d>
> reconstruction_segments;

for (int i = 0;
     i < X.rows;
     ++i)
{
    reconstruction_segments.push_back({
        pca_points[i],
        reconstructed_points[i]
    });
}

cv::Mat reconstruction_plot =
    render_xy_plot(
        pca_points,
        reconstructed_points,
        reconstruction_segments,
        "Informationsverlust bei 2D -> 1D",
        "Feature 1",
        "Feature 2"
    );

In [ ]:
show_image(reconstruction_plot)

## 9. PCA auf Bilddaten

Nun wenden wir dieselbe Idee auf echte Bilddaten an.

Die Python-Version verwendet den `digits`-Datensatz aus `scikit-learn`. Für die C++-Version verwenden wir stattdessen das offizielle OpenCV-Testbild **`digits.png`**.

Es enthält **5000 handgeschriebene Ziffern**, jeweils als **20 × 20 Pixel** großes Grauwertbild.

Ein Bild wird zu einem Vektor abgeflacht:

$20\times20 = 400$

Damit liegt jedes Bild zunächst in einem **400-dimensionalen Raum**.

Die PCA-Idee ist exakt dieselbe wie in der Python-Version; lediglich der verwendete Bilddatensatz ist C++-/OpenCV-nativer.

In [ ]:
download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/digits.png",
    "digits.png"
);

cv::Mat digits_image =
    cv::imread(
        "digits.png",
        cv::IMREAD_GRAYSCALE
    );

if (digits_image.empty())
{
    std::cerr
        << "digits.png konnte nicht geladen werden."
        << std::endl;
}

std::cout
    << "Gesamtbild: "
    << digits_image.cols
    << " x "
    << digits_image.rows
    << std::endl;

### Datensatz aus dem Gesamtbild extrahieren

Das OpenCV-Testbild ist als Raster aufgebaut:

- 100 Ziffern pro Zeile,
- 50 Zeilen,
- jede Ziffer 20 × 20 Pixel,
- jeweils fünf Bildzeilen gehören zu derselben Ziffernklasse.

Daraus erzeugen wir eine Matrix $\mathbf{X}\in\mathbb{R}^{5000\times400}$.

In [ ]:
const int digit_size = 20;
const int digit_columns = 100;
const int digit_rows = 50;
const int digit_count =
    digit_columns * digit_rows;
const int digit_dimension =
    digit_size * digit_size;

cv::Mat X_images(
    digit_count,
    digit_dimension,
    CV_32F
);

cv::Mat digit_labels(
    digit_count,
    1,
    CV_32S
);

int digit_index = 0;

for (int row = 0;
     row < digit_rows;
     ++row)
{
    int label =
        row / 5;

    for (int column = 0;
         column < digit_columns;
         ++column)
    {
        cv::Rect roi(
            column * digit_size,
            row * digit_size,
            digit_size,
            digit_size
        );

        cv::Mat digit =
            digits_image(
                roi
            );

        int feature_index = 0;

        for (int y_pixel = 0;
             y_pixel < digit_size;
             ++y_pixel)
        {
            for (int x_pixel = 0;
                 x_pixel < digit_size;
                 ++x_pixel)
            {
                X_images.at<float>(
                    digit_index,
                    feature_index
                ) =
                    static_cast<float>(
                        digit.at<unsigned char>(
                            y_pixel,
                            x_pixel
                        )
                    );

                feature_index++;
            }
        }

        digit_labels.at<int>(
            digit_index,
            0
        ) =
            label;

        digit_index++;
    }
}

std::cout << "Anzahl Bilder: "
          << X_images.rows
          << std::endl;

std::cout << "Feature-Dimension: "
          << X_images.cols
          << std::endl;

### Hilfsfunktion zur Darstellung einzelner Ziffern

In [ ]:
cv::Mat render_digit_examples(
    const cv::Mat& data,
    const std::vector<int>& indices,
    int digit_size
)
{
    const int padding = 8;
    const int scale = 5;

    int width =
        static_cast<int>(
            indices.size()
        )
        * (
            digit_size * scale
            + padding
        )
        + padding;

    int height =
        digit_size * scale
        + 2 * padding
        + 30;

    cv::Mat canvas(
        height,
        width,
        CV_8UC1,
        cv::Scalar(255)
    );

    for (std::size_t i = 0;
         i < indices.size();
         ++i)
    {
        int index =
            indices[i];

        cv::Mat digit(
            digit_size,
            digit_size,
            CV_32F
        );

        for (int y = 0;
             y < digit_size;
             ++y)
        {
            for (int x = 0;
                 x < digit_size;
                 ++x)
            {
                digit.at<float>(
                    y,
                    x
                ) =
                    data.at<float>(
                        index,
                        y * digit_size + x
                    );
            }
        }

        cv::Mat digit_u8;

        digit.convertTo(
            digit_u8,
            CV_8U
        );

        cv::Mat enlarged;

        cv::resize(
            digit_u8,
            enlarged,
            cv::Size(),
            scale,
            scale,
            cv::INTER_NEAREST
        );

        int x_offset =
            padding
            + static_cast<int>(i)
            * (
                digit_size * scale
                + padding
            );

        enlarged.copyTo(
            canvas(
                cv::Rect(
                    x_offset,
                    padding,
                    enlarged.cols,
                    enlarged.rows
                )
            )
        );
    }

    return canvas;
}

In [ ]:
std::vector<int>
    digit_example_indices = {
        0,
        500,
        1000,
        1500,
        2000,
        2500,
        3000,
        3500
    };

cv::Mat digit_examples =
    render_digit_examples(
        X_images,
        digit_example_indices,
        digit_size
    );

In [ ]:
show_image(digit_examples)

### 400 Dimensionen → 2 Dimensionen

PCA wird **ohne Labels** trainiert.

Die Ziffernlabels verwenden wir anschließend nur zur Visualisierung, damit sichtbar wird, ob sich im reduzierten Raum Strukturen bilden.

In [ ]:
cv::PCA pca_2d(
    X_images,
    cv::Mat(),
    cv::PCA::DATA_AS_ROW,
    2
);

cv::Mat X_pca_2d;

pca_2d.project(
    X_images,
    X_pca_2d
);

std::cout
    << "Reduzierte Dimension: "
    << X_pca_2d.rows
    << " x "
    << X_pca_2d.cols
    << std::endl;

std::cout
    << "Eigenwerte der ersten beiden Komponenten:"
    << std::endl
    << pca_2d.eigenvalues
    << std::endl;

### Hilfsfunktion für den 2D-PCA-Raum der Ziffern

In [ ]:
cv::Mat render_digit_pca_scatter(
    const cv::Mat& projected,
    const cv::Mat& labels,
    const std::string& title
)
{
    const int width = 760;
    const int height = 560;
    const int left = 70;
    const int right = 30;
    const int top = 45;
    const int bottom = 60;

    double min_x =
        projected.at<float>(0, 0);

    double max_x =
        min_x;

    double min_y =
        projected.at<float>(0, 1);

    double max_y =
        min_y;

    for (int i = 0;
         i < projected.rows;
         ++i)
    {
        min_x = std::min(
            min_x,
            static_cast<double>(
                projected.at<float>(
                    i,
                    0
                )
            )
        );

        max_x = std::max(
            max_x,
            static_cast<double>(
                projected.at<float>(
                    i,
                    0
                )
            )
        );

        min_y = std::min(
            min_y,
            static_cast<double>(
                projected.at<float>(
                    i,
                    1
                )
            )
        );

        max_y = std::max(
            max_y,
            static_cast<double>(
                projected.at<float>(
                    i,
                    1
                )
            )
        );
    }

    double range_x =
        std::max(
            max_x - min_x,
            1e-9
        );

    double range_y =
        std::max(
            max_y - min_y,
            1e-9
        );

    min_x -=
        0.05 * range_x;

    max_x +=
        0.05 * range_x;

    min_y -=
        0.05 * range_y;

    max_y +=
        0.05 * range_y;

    range_x =
        max_x - min_x;

    range_y =
        max_y - min_y;

    auto to_pixel =
        [&](double x, double y)
        {
            int px = cvRound(
                left
                + (x - min_x)
                / range_x
                * (width - left - right)
            );

            int py = cvRound(
                height - bottom
                - (y - min_y)
                / range_y
                * (height - top - bottom)
            );

            return cv::Point(
                px,
                py
            );
        };

    std::vector<cv::Scalar> colors = {
        cv::Scalar(230, 80, 70),
        cv::Scalar(70, 150, 230),
        cv::Scalar(70, 180, 90),
        cv::Scalar(180, 90, 200),
        cv::Scalar(40, 180, 200),
        cv::Scalar(200, 150, 40),
        cv::Scalar(110, 110, 110),
        cv::Scalar(160, 100, 50),
        cv::Scalar(90, 180, 170),
        cv::Scalar(200, 90, 140)
    };

    cv::Mat canvas(
        height,
        width,
        CV_8UC3,
        cv::Scalar(255, 255, 255)
    );

    cv::line(
        canvas,
        cv::Point(left, top),
        cv::Point(left, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    cv::line(
        canvas,
        cv::Point(left, height - bottom),
        cv::Point(width - right, height - bottom),
        cv::Scalar(0, 0, 0),
        2
    );

    for (int i = 0;
         i < projected.rows;
         ++i)
    {
        int label =
            labels.at<int>(
                i,
                0
            );

        cv::circle(
            canvas,
            to_pixel(
                projected.at<float>(
                    i,
                    0
                ),
                projected.at<float>(
                    i,
                    1
                )
            ),
            2,
            colors[
                label % colors.size()
            ],
            -1,
            cv::LINE_AA
        );
    }

    cv::putText(
        canvas,
        title,
        cv::Point(20, 28),
        cv::FONT_HERSHEY_SIMPLEX,
        0.65,
        cv::Scalar(0, 0, 0),
        2
    );

    return canvas;
}

In [ ]:
cv::Mat pca_digit_plot =
    render_digit_pca_scatter(
        X_pca_2d,
        digit_labels,
        "400D-Bilddaten -> 2D-PCA-Raum"
    );

In [ ]:
show_image(pca_digit_plot)

PCA kennt die Klassen nicht. Sie sucht **Varianz**, nicht automatisch semantisch relevante Information.

In [ ]:
cv::PCA pca_full(
    X_images,
    cv::Mat(),
    cv::PCA::DATA_AS_ROW,
    X_images.cols
);

double total_eigenvalue_sum =
    cv::sum(
        pca_full.eigenvalues
    )[0];

std::vector<cv::Point2d>
    cumulative_variance_points;

double cumulative_sum = 0.0;

for (int i = 0;
     i < pca_full.eigenvalues.rows;
     ++i)
{
    cumulative_sum +=
        pca_full.eigenvalues.at<float>(
            i,
            0
        );

    cumulative_variance_points.emplace_back(
        i + 1,
        cumulative_sum
            / total_eigenvalue_sum
    );
}

cv::Mat cumulative_variance_plot =
    render_xy_plot(
        {},
        cumulative_variance_points,
        {},
        "PCA: Informationsgehalt vs. Dimension",
        "Anzahl Hauptkomponenten",
        "Kumulierte Varianz"
    );

In [ ]:
show_image(cumulative_variance_plot)

### Hilfsfunktion für Original und PCA-Rekonstruktion

In [ ]:
cv::Mat render_digit_comparison(
    const cv::Mat& original,
    const cv::Mat& reconstructed,
    const std::vector<int>& indices,
    int digit_size
)
{
    const int padding = 8;
    const int scale = 4;

    int cell_size =
        digit_size * scale;

    int width =
        static_cast<int>(
            indices.size()
        )
        * (
            cell_size
            + padding
        )
        + padding;

    int height =
        2 * cell_size
        + 3 * padding
        + 25;

    cv::Mat canvas(
        height,
        width,
        CV_8UC1,
        cv::Scalar(255)
    );

    for (std::size_t i = 0;
         i < indices.size();
         ++i)
    {
        int index =
            indices[i];

        for (int row_group = 0;
             row_group < 2;
             ++row_group)
        {
            const cv::Mat& data =
                row_group == 0
                ? original
                : reconstructed;

            cv::Mat digit(
                digit_size,
                digit_size,
                CV_32F
            );

            for (int y = 0;
                 y < digit_size;
                 ++y)
            {
                for (int x = 0;
                     x < digit_size;
                     ++x)
                {
                    digit.at<float>(
                        y,
                        x
                    ) =
                        data.at<float>(
                            index,
                            y * digit_size + x
                        );
                }
            }

            cv::Mat clipped;

            cv::max(
                digit,
                0,
                clipped
            );

            cv::min(
                clipped,
                255,
                clipped
            );

            cv::Mat digit_u8;

            clipped.convertTo(
                digit_u8,
                CV_8U
            );

            cv::Mat enlarged;

            cv::resize(
                digit_u8,
                enlarged,
                cv::Size(),
                scale,
                scale,
                cv::INTER_NEAREST
            );

            int x_offset =
                padding
                + static_cast<int>(i)
                * (
                    cell_size
                    + padding
                );

            int y_offset =
                padding
                + row_group
                * (
                    cell_size
                    + padding
                );

            enlarged.copyTo(
                canvas(
                    cv::Rect(
                        x_offset,
                        y_offset,
                        enlarged.cols,
                        enlarged.rows
                    )
                )
            );
        }
    }

    cv::putText(
        canvas,
        "oben: Original | unten: PCA-Rekonstruktion",
        cv::Point(
            10,
            height - 8
        ),
        cv::FONT_HERSHEY_SIMPLEX,
        0.55,
        cv::Scalar(0),
        1
    );

    return canvas;
}

### Mini-Aufgabe: PCA-Dimension

Verändern Sie `n_components_test` und beobachten Sie die Rekonstruktionsqualität.

In [ ]:
[]()
{
    // TODO: Anzahl der Hauptkomponenten verändern

    int n_components_test = 20;

    cv::PCA pca_test(
        X_images,
        cv::Mat(),
        cv::PCA::DATA_AS_ROW,
        n_components_test
    );

    cv::Mat X_reduced;

    pca_test.project(
        X_images,
        X_reduced
    );

    cv::Mat X_reconstructed_images;

    pca_test.backProject(
        X_reduced,
        X_reconstructed_images
    );

    double retained_eigenvalue_sum =
        cv::sum(
            pca_test.eigenvalues
        )[0];

    double explained_fraction =
        retained_eigenvalue_sum
        / total_eigenvalue_sum;

    std::cout
        << "Dimension vorher: "
        << X_images.cols
        << std::endl;

    std::cout
        << "Dimension nachher: "
        << X_reduced.cols
        << std::endl;

    std::cout
        << "Erklaerte Varianz: "
        << explained_fraction
        << std::endl;

    cv::Mat result =
        render_digit_comparison(
            X_images,
            X_reconstructed_images,
            digit_example_indices,
            digit_size
        );

    return show_image(result);
}()

## 10. Von PCA zu gelernten Bildmerkmalen

PCA lernt bereits eine Repräsentation aus Daten:

**hochdimensionale Daten → gelernte lineare Projektion → kompakter Merkmalsraum**

Die Projektion ist jedoch **linear** und optimiert auf Varianz.

Moderne Foundation Models wie DINO lernen wesentlich komplexere Repräsentationen. Ein häufiger Visualisierungsschritt ist anschließend wieder PCA:

**hochdimensionale gelernte Features → PCA auf 3 Dimensionen → RGB-Visualisierung**

Eine vollständige DINO-Pipeline ist in diesem Vorbereitungsnotebook bewusst nicht enthalten, da dafür ein großer Modell-Download und ein deutlich umfangreicheres Deep-Learning-Setup nötig wären.

## 11. Selbstcheck

1. Was unterscheidet Supervised und Unsupervised Learning?
2. Was ist der Unterschied zwischen Klassifikation und Regression?
3. Welche Größen werden bei einer linearen Regression gelernt?
4. Was ist ein Residuum?
5. Warum werden Fehler beim MSE quadriert?
6. Welche Rolle spielt die Learning Rate?
7. Was bedeutet Generalisierung?
8. Was bezeichnet $k$ bei k-Means?
9. Wie werden Datenpunkte bei k-Means einem Cluster zugeordnet?
10. Warum kann Feature-Skalierung für k-Means wichtig sein?
11. Was ist das Ziel der PCA?
12. Welche Bedeutung besitzen Eigenvektoren und Eigenwerte bei PCA?
13. Optimiert PCA direkt auf semantisch relevante Merkmale?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. Supervised Learning verwendet Labels, Unsupervised Learning nicht.
2. Klassifikation sagt diskrete Klassen voraus, Regression kontinuierliche Werte.
3. Steigung $w$ und Achsenabschnitt $b$.
4. Die Differenz zwischen tatsächlichem und vorhergesagtem Wert.
5. Damit sich positive und negative Fehler nicht aufheben und große Fehler stärker eingehen.
6. Sie bestimmt die Größe der Parameterschritte beim Training.
7. Gute Vorhersagen auf neuen, nicht im Training gesehenen Daten.
8. Die Anzahl der Cluster.
9. Über die Distanz zum nächstgelegenen Clusterzentrum.
10. Sonst dominieren Features mit großen Zahlenbereichen die Distanz.
11. Dimensionsreduktion bei möglichst großem Erhalt der Variation.
12. Eigenvektoren liefern die Projektionsrichtungen, Eigenwerte die zugehörige Varianz.
13. Nein. PCA maximiert Varianz.

</details>

## 12. Take-away

- Machine Learning lernt Abhängigkeiten aus Daten, statt sie vollständig von Hand vorzugeben.
- Supervised Learning verwendet Eingaben und Ground-Truth-Labels.
- Regression sagt kontinuierliche Werte voraus.
- Ein Loss macht die Qualität eines Modells quantitativ messbar.
- Gradient Descent passt Modellparameter iterativ an.
- Ein gutes Modell soll auf neue Daten **generalisieren**.
- Unsupervised Learning arbeitet ohne vorgegebene Labels.
- k-Means gruppiert Daten anhand ihrer Distanz zu Clusterzentren.
- Bei k-Means bezeichnet $k$ die **Anzahl der Cluster**.
- PCA lernt eine lineare Projektion in Richtungen maximaler Varianz.
- PCA kann hochdimensionale Bilddaten kompakter darstellen.
- PCA reduziert Dimension, kann dabei aber Information verlieren.

### Ausblick

Die Struktur

**Daten → Modell → Loss/Zielkriterium → Optimierung → gelernte Repräsentation oder Vorhersage**

taucht auch bei Deep-Learning-Verfahren wieder auf.